In [148]:
# This notebook constructs MEPI indicators, calculates MEPI scores, and derives target variables using weighted assignments
import pandas as pd
import numpy as np

df = pd.read_parquet("INDIA/DATA/1.df_IN_NecessaryColumns.parquet", engine="pyarrow")

# 1) Basic shape
print("Rows, Cols:", df.shape)

# 2) Household ID checks
print("\nID missing:", df["hhidCaseIdentification"].isna().sum())
print("ID duplicates:", df["hhidCaseIdentification"].duplicated().sum())

# Show duplicate examples (if any)
dups = df[df["hhidCaseIdentification"].duplicated(keep=False)].sort_values("hhidCaseIdentification")
print("\nDuplicate ID sample (first 10 rows):")
print(dups[["hhidCaseIdentification","hv000CountryCode","hv001ClusterNumber","hv002HouseholdNumber"]].head(10))


Rows, Cols: (636699, 29)

ID missing: 0
ID duplicates: 0

Duplicate ID sample (first 10 rows):
Empty DataFrame
Columns: [hhidCaseIdentification, hv000CountryCode, hv001ClusterNumber, hv002HouseholdNumber]
Index: []


In [149]:
print(df["hv270WealthIndex"].value_counts(dropna=False).sort_index())
print("dtype:", df["hv270WealthIndex"].dtype)
#1    2608  → Poorest
#2    2627  → Poorer
#3    2525  → Middle
#4    2458  → Richer
#5    2282  → Richest
#dtype: int8

hv270WealthIndex
1    148772
2    141152
3    129057
4    115002
5    102716
Name: count, dtype: int64
dtype: int8


In [150]:
print(df["hv206HasElectricity"].value_counts(dropna=False))
print("dtype:", df["hv206HasElectricity"].dtype)

hv206HasElectricity
1    614107
0     22592
Name: count, dtype: int64
dtype: int8


In [151]:
print(df["hv226CookingFuel"].value_counts(dropna=False))
print("dtype:", df["hv226CookingFuel"].dtype)
#{"1": "electricity", "2": "lpg", "3": "natural gas", "4": "biogas", "5": "kerosene",
 #"6": "coal, lignite", "7": "charcoal", "8": "wood", "9": "straw/shrubs/grass", 
 #"10": "agricultural crop", "11": "animal dung", "95": "no food cooked in house", "96": "other"}

hv226CookingFuel
2     327347
8     246040
11     23537
10      9836
1       7282
9       5695
7       5081
6       4919
5       3051
4       2204
95      1075
96       632
Name: count, dtype: int64
dtype: int8


In [152]:
print(df["hv242SeparateKitchenYesNo"].value_counts(dropna=False))
print("dtype:", df["hv242SeparateKitchenYesNo"].dtype)

hv242SeparateKitchenYesNo
1.0    351739
0.0    148886
NaN    136074
Name: count, dtype: int64
dtype: float64


In [153]:
print(df["hv241FoodCookedInHouse"].value_counts(dropna=False))
print("dtype:", df["hv241FoodCookedInHouse"].dtype)

hv241FoodCookedInHouse
1.0    500625
2.0     88695
3.0     45781
NaN      1075
6.0       523
Name: count, dtype: int64
dtype: float64


In [154]:
print(df["hv209HasRefrigerator"].value_counts(dropna=False))
print("dtype:", df["hv209HasRefrigerator"].dtype)

hv209HasRefrigerator
0    416229
1    220470
Name: count, dtype: int64
dtype: int8


In [155]:
print(df["hv208HasTelevision"].value_counts(dropna=False))
print("dtype:", df["hv208HasTelevision"].dtype)

hv208HasTelevision
1    418778
0    217921
Name: count, dtype: int64
dtype: int8


In [156]:
print(df["hv243aHasMobilePhone"].value_counts(dropna=False))
print("dtype:", df["hv243aHasMobilePhone"].dtype)

hv243aHasMobilePhone
1    589135
0     47564
Name: count, dtype: int64
dtype: int8


In [157]:
print(df["hv221HasLandLine"].value_counts(dropna=False))
print("dtype:", df["hv221HasLandLine"].dtype)

hv221HasLandLine
0    624944
1     11755
Name: count, dtype: int64
dtype: int8


In [158]:
# 2) Create deprivation indicators (1=deprived, 0=not deprived)
# -------------------------
# Electricity (1=yes, 0=no)
df["Depr_Elec"] = np.where(df["hv206HasElectricity"] == 1, 0,
                    np.where(df["hv206HasElectricity"] == 0, 1, np.nan))

In [159]:
list(df.columns)

['hhidCaseIdentification',
 'hv000CountryCode',
 'hv001ClusterNumber',
 'hv002HouseholdNumber',
 'hv005SimplilingWeight',
 'hv270WealthIndex',
 'hv206HasElectricity',
 'hv226CookingFuel',
 'hv241FoodCookedInHouse',
 'hv242SeparateKitchenYesNo',
 'hv209HasRefrigerator',
 'hv221HasLandLine',
 'hv243aHasMobilePhone',
 'hv208HasTelevision',
 'hv024RegionDivision',
 'hv025TypeOfResidence',
 'hv219SexOfHead',
 'hv220AgeOfHead',
 'hv106_01EducationOfHead',
 'hv115_01MaritalStatus',
 'hv009FamilySize',
 'hv247HasBankAccount',
 'hv216HouseSize',
 'hv014NoOfChildren',
 'v714CurrentlyWorking',
 'v717Occupation',
 '745aHouseOwnership',
 'weight',
 'HouseholdClusterElevation',
 'Depr_Elec']

In [160]:
print(df["hv206HasElectricity"].value_counts(dropna=False))
print("dtype:", df["hv206HasElectricity"].dtype)

hv206HasElectricity
1    614107
0     22592
Name: count, dtype: int64
dtype: int8


In [161]:
print(df["Depr_Elec"].value_counts(dropna=False))
print("dtype:", df["Depr_Elec"].dtype)

Depr_Elec
0.0    614107
1.0     22592
Name: count, dtype: int64
dtype: float64


In [162]:
# Cooking fuel: NOT deprived if using electricity, gas (LPG+natural gas), kerosene, or biogas
CLEAN_FUEL_CODES = {1, 2, 3, 4, 5}

# Cooking fuel (clean -> 0, other -> 1)
df["Depr_CleanFuel"] = np.where(df["hv226CookingFuel"].isin(CLEAN_FUEL_CODES), 0,
                         np.where(df["hv226CookingFuel"].isna(), np.nan, 1))

In [163]:
#print(df["hv226CookingFuel"].value_counts(dropna=False))
#print("dtype:", df["hv226CookingFuel"].dtype)
CLEAN_FUEL_CODES = {1, 2, 3, 4, 5}
print (df["hv226CookingFuel"].isin(CLEAN_FUEL_CODES).sum())
print (df["hv226CookingFuel"].isin(CLEAN_FUEL_CODES).eq(False).sum())
#{"1": "electricity", "2": "lpg", "3": "natural gas", "4": "biogas", "5": "kerosene",
 #"6": "coal, lignite", "7": "charcoal", "8": "wood", "9": "straw/shrubs/grass", 
 #"10": "agricultural crop", "11": "animal dung", "95": "no food cooked in house", "96": "other"}

print(df["Depr_CleanFuel"].value_counts(dropna=False))
print("dtype:", df["Depr_CleanFuel"].dtype)

339884
296815
Depr_CleanFuel
0.0    339884
1.0    296815
Name: count, dtype: int64
dtype: float64


In [164]:
# Refrigerator
df["Depr_Refrigerator"] = np.where(df["hv209HasRefrigerator"] == 1, 0,
                            np.where(df["hv209HasRefrigerator"] == 0, 1, np.nan))

In [165]:
print(df["hv209HasRefrigerator"].value_counts(dropna=False))
print("dtype:", df["hv209HasRefrigerator"].dtype)

print(df["Depr_Refrigerator"].value_counts(dropna=False))
print("dtype:", df["Depr_Refrigerator"].dtype)

hv209HasRefrigerator
0    416229
1    220470
Name: count, dtype: int64
dtype: int8
Depr_Refrigerator
1.0    416229
0.0    220470
Name: count, dtype: int64
dtype: float64


In [166]:
# Television
df["Depr_TV"] = np.where(df["hv208HasTelevision"] == 1, 0,
                  np.where(df["hv208HasTelevision"] == 0, 1, np.nan))

In [167]:
print(df["hv208HasTelevision"].value_counts(dropna=False))
print("dtype:", df["hv208HasTelevision"].dtype)

print(df["Depr_TV"].value_counts(dropna=False))
print("dtype:", df["Depr_TV"].dtype)

hv208HasTelevision
1    418778
0    217921
Name: count, dtype: int64
dtype: int8
Depr_TV
0.0    418778
1.0    217921
Name: count, dtype: int64
dtype: float64


In [168]:
# Phone (deprived only if NO mobile AND NO landline)
df["Depr_Phone"] = np.where(
    df["hv243aHasMobilePhone"].isna() | df["hv221HasLandLine"].isna(),
    np.nan,
    np.where((df["hv243aHasMobilePhone"] == 0) & (df["hv221HasLandLine"] == 0), 1, 0)
)

In [169]:
list(df.columns)

['hhidCaseIdentification',
 'hv000CountryCode',
 'hv001ClusterNumber',
 'hv002HouseholdNumber',
 'hv005SimplilingWeight',
 'hv270WealthIndex',
 'hv206HasElectricity',
 'hv226CookingFuel',
 'hv241FoodCookedInHouse',
 'hv242SeparateKitchenYesNo',
 'hv209HasRefrigerator',
 'hv221HasLandLine',
 'hv243aHasMobilePhone',
 'hv208HasTelevision',
 'hv024RegionDivision',
 'hv025TypeOfResidence',
 'hv219SexOfHead',
 'hv220AgeOfHead',
 'hv106_01EducationOfHead',
 'hv115_01MaritalStatus',
 'hv009FamilySize',
 'hv247HasBankAccount',
 'hv216HouseSize',
 'hv014NoOfChildren',
 'v714CurrentlyWorking',
 'v717Occupation',
 '745aHouseOwnership',
 'weight',
 'HouseholdClusterElevation',
 'Depr_Elec',
 'Depr_CleanFuel',
 'Depr_Refrigerator',
 'Depr_TV',
 'Depr_Phone']

In [170]:
print(df["Depr_Phone"].value_counts(dropna=False))
print("dtype:", df["Depr_Phone"].dtype)

Depr_Phone
0.0    589972
1.0     46727
Name: count, dtype: int64
dtype: float64


In [171]:
# Kitchen (hv242 preferred; fallback hv241FoodCookedInHouse)
df["Depr_Kitchen"] = np.nan

# hv242SeparateKitchenYesNo: 1 yes (not deprived), 0 no (deprived)
df.loc[df["hv242SeparateKitchenYesNo"] == 1, "Depr_Kitchen"] = 0
df.loc[df["hv242SeparateKitchenYesNo"] == 0, "Depr_Kitchen"] = 1

# fallback to hv241FoodCookedInHouse when hv242 missing
mask = df["hv242SeparateKitchenYesNo"].isna()

In [172]:
print(df["Depr_Kitchen"].value_counts(dropna=False))
print("dtype:", df["Depr_Kitchen"].dtype)

#print(mask)

Depr_Kitchen
0.0    351739
1.0    148886
NaN    136074
Name: count, dtype: int64
dtype: float64


In [173]:
# hv241FoodCookedInHouse: 2 separate building -> not deprived
df.loc[mask & (df["hv241FoodCookedInHouse"] == 2), "Depr_Kitchen"] = 0
# 1 in the house, 3 outdoors, 6 other -> deprived
df.loc[mask & df["hv241FoodCookedInHouse"].isin([1, 3, 6]), "Depr_Kitchen"] = 1

In [174]:
print(df["Depr_Kitchen"].value_counts(dropna=False))
print("dtype:", df["Depr_Kitchen"].dtype)

Depr_Kitchen
0.0    440434
1.0    195190
NaN      1075
Name: count, dtype: int64
dtype: float64


In [175]:
list(df.columns)

['hhidCaseIdentification',
 'hv000CountryCode',
 'hv001ClusterNumber',
 'hv002HouseholdNumber',
 'hv005SimplilingWeight',
 'hv270WealthIndex',
 'hv206HasElectricity',
 'hv226CookingFuel',
 'hv241FoodCookedInHouse',
 'hv242SeparateKitchenYesNo',
 'hv209HasRefrigerator',
 'hv221HasLandLine',
 'hv243aHasMobilePhone',
 'hv208HasTelevision',
 'hv024RegionDivision',
 'hv025TypeOfResidence',
 'hv219SexOfHead',
 'hv220AgeOfHead',
 'hv106_01EducationOfHead',
 'hv115_01MaritalStatus',
 'hv009FamilySize',
 'hv247HasBankAccount',
 'hv216HouseSize',
 'hv014NoOfChildren',
 'v714CurrentlyWorking',
 'v717Occupation',
 '745aHouseOwnership',
 'weight',
 'HouseholdClusterElevation',
 'Depr_Elec',
 'Depr_CleanFuel',
 'Depr_Refrigerator',
 'Depr_TV',
 'Depr_Phone',
 'Depr_Kitchen']

In [176]:
df = df.dropna(subset=["Depr_Kitchen"])

In [177]:
print(df["Depr_Kitchen"].value_counts(dropna=False))
print("dtype:", df["Depr_Kitchen"].dtype)

Depr_Kitchen
0.0    440434
1.0    195190
Name: count, dtype: int64
dtype: float64


In [178]:
weights = {
    "Depr_Elec": 0.20,
    "Depr_CleanFuel": 0.20,
    "Depr_Kitchen": 0.15,
    "Depr_Refrigerator": 0.15,
    "Depr_TV": 0.15,
    "Depr_Phone": 0.15
  }

In [179]:
mepi_cols = list(weights.keys())
print(mepi_cols)

['Depr_Elec', 'Depr_CleanFuel', 'Depr_Kitchen', 'Depr_Refrigerator', 'Depr_TV', 'Depr_Phone']


In [180]:
# Convert to numeric 0/1 (in case they are 'yes'/'no' or categories)
for col in mepi_cols:
    df[col] = df[col].astype(int)

# Quick check
for col in mepi_cols:
    print(col)
    print(df[col].value_counts(dropna=False), "\n")


Depr_Elec
Depr_Elec
0    613143
1     22481
Name: count, dtype: int64 

Depr_CleanFuel
Depr_CleanFuel
0    339884
1    295740
Name: count, dtype: int64 

Depr_Kitchen
Depr_Kitchen
0    440434
1    195190
Name: count, dtype: int64 

Depr_Refrigerator
Depr_Refrigerator
1    415257
0    220367
Name: count, dtype: int64 

Depr_TV
Depr_TV
0    418466
1    217158
Name: count, dtype: int64 

Depr_Phone
Depr_Phone
0    589325
1     46299
Name: count, dtype: int64 



In [181]:
df["MEPI_score"] = np.nan
df['MEPI_score'] = sum(df[col] * w for col, w in weights.items())

In [182]:
print(df["MEPI_score"].value_counts(dropna=False))

MEPI_score
0.00    152696
0.15    105437
0.50     99867
0.35     74709
0.65     68190
0.30     54106
0.20     24514
0.45     21631
0.80     12797
0.85      9377
0.70      5314
1.00      3885
0.60      2605
0.55       442
0.40        54
Name: count, dtype: int64


In [183]:
# get dependent varialble Energy Poor
ROUND_DECIMALS = 4  # Adjust based on the precision needs
CUTOFF = 0.35

df['EnergyPoor'] = (df['MEPI_score'].round(ROUND_DECIMALS) >= CUTOFF).astype(float)
df['EnergyPoor'] = df['EnergyPoor'].where(df['MEPI_score'].notna(), np.nan)

# Verification
print(f"Energy Poor: {(df['EnergyPoor'] == 1).sum():,}")
print(f"Not Energy Poor: {(df['EnergyPoor'] == 0).sum():,}")
print(f"Missing: {df['EnergyPoor'].isna().sum():,}")

# Should output:
# Energy Poor: 298,871
# Not Energy Poor: 336,753
# Missing: [whatever missing MEPI scores you have]

Energy Poor: 298,871
Not Energy Poor: 336,753
Missing: 0


In [184]:
list(df.columns)

['hhidCaseIdentification',
 'hv000CountryCode',
 'hv001ClusterNumber',
 'hv002HouseholdNumber',
 'hv005SimplilingWeight',
 'hv270WealthIndex',
 'hv206HasElectricity',
 'hv226CookingFuel',
 'hv241FoodCookedInHouse',
 'hv242SeparateKitchenYesNo',
 'hv209HasRefrigerator',
 'hv221HasLandLine',
 'hv243aHasMobilePhone',
 'hv208HasTelevision',
 'hv024RegionDivision',
 'hv025TypeOfResidence',
 'hv219SexOfHead',
 'hv220AgeOfHead',
 'hv106_01EducationOfHead',
 'hv115_01MaritalStatus',
 'hv009FamilySize',
 'hv247HasBankAccount',
 'hv216HouseSize',
 'hv014NoOfChildren',
 'v714CurrentlyWorking',
 'v717Occupation',
 '745aHouseOwnership',
 'weight',
 'HouseholdClusterElevation',
 'Depr_Elec',
 'Depr_CleanFuel',
 'Depr_Refrigerator',
 'Depr_TV',
 'Depr_Phone',
 'Depr_Kitchen',
 'MEPI_score',
 'EnergyPoor']

In [185]:
df.to_parquet(
    "INDIA/DATA/2.df_IN_MEPI_Score_Energy_Poor.parquet",
    engine="pyarrow",
    compression="snappy"
)
print(f"✅ File saved.")

✅ File saved.
